# Public Health RFP Evaluation Analytics

Reads evaluation results from OneLake and produces Power BI-ready tables:

| Table | Contents |
|---|---|
| `eval_results` | Raw eval pairs from golden dataset |
| `pass_rates_by_program` | Aggregated pass/fail by program area |
| `score_distributions` | Percentile distribution of each evaluator score |
| `gate_confusion` | TP/TN/FP/FN counts for gate calibration |
| `budget_summary` | Monthly spend vs. threshold from session tracker |

All tables are written to the default Lakehouse as Delta tables and auto-appear in the Power BI semantic model.

In [ ]:
import json
from datetime import datetime

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    DoubleType, IntegerType, StringType, StructField, StructType, TimestampType
)

spark = SparkSession.builder.getOrCreate()

ONELAKE_PATH = "Files/golden-dataset/"
EVAL_RESULTS_PATH = f"{ONELAKE_PATH}eval_pairs.jsonl"
CALIBRATION_PATH = f"{ONELAKE_PATH}calibration_report.json"

DIMENSIONS = ["groundedness", "completeness", "parameter_accuracy", "compliance", "coherence"]

print("✓ Imports OK")

In [ ]:
# ── Load eval pairs from OneLake ──────────────────────────────────────────────
eval_schema = StructType([
    StructField("eval_pair_id", StringType()),
    StructField("rfp_id", StringType()),
    StructField("program_area", StringType()),
    StructField("annotator_id", StringType()),
    StructField("annotated_at", StringType()),
    StructField("human_gate", StringType()),
    StructField("model_gate", StringType()),
    StructField("gate_agreement", StringType()),
    StructField("confusion_cell", StringType()),
    *[StructField(f"human_{d}", DoubleType()) for d in DIMENSIONS],
    *[StructField(f"model_{d}", DoubleType()) for d in DIMENSIONS],
    *[StructField(f"delta_{d}", DoubleType()) for d in DIMENSIONS],
])

try:
    df_eval = spark.read.schema(eval_schema).json(EVAL_RESULTS_PATH)
    df_eval = df_eval.withColumn(
        "annotated_at",
        F.to_timestamp("annotated_at", "yyyy-MM-dd'T'HH:mm:ss")
    ).withColumn(
        "gate_agreement", F.col("gate_agreement").cast("boolean")
    )
    total = df_eval.count()
    print(f"Loaded {total} eval pairs from OneLake")
    df_eval.printSchema()
except Exception as e:
    print(f"⚠ Could not load {EVAL_RESULTS_PATH}: {e}")
    print("Creating empty frame for schema development — populate via annotator_cli.py + eval_pairs.py")
    df_eval = spark.createDataFrame([], eval_schema)

In [ ]:
# ── Table 1: eval_results (base table) ──────────────────────────────────────
df_eval.write.format("delta").mode("overwrite").saveAsTable("eval_results")
print("✓ eval_results saved")

In [ ]:
# ── Table 2: pass_rates_by_program ───────────────────────────────────────────
df_pass = (
    df_eval
    .groupBy("program_area")
    .agg(
        F.count("*").alias("total_drafts"),
        F.sum(F.when(F.col("human_gate") == "PASS", 1).otherwise(0)).alias("human_pass"),
        F.sum(F.when(F.col("model_gate") == "PASS", 1).otherwise(0)).alias("model_pass"),
        *[
            F.round(F.avg(f"human_{d}"), 3).alias(f"avg_human_{d}")
            for d in DIMENSIONS
        ],
        *[
            F.round(F.avg(f"model_{d}"), 3).alias(f"avg_model_{d}")
            for d in DIMENSIONS
        ],
    )
    .withColumn("human_pass_rate", F.round(F.col("human_pass") / F.col("total_drafts"), 3))
    .withColumn("model_pass_rate", F.round(F.col("model_pass") / F.col("total_drafts"), 3))
    .orderBy("human_pass_rate", ascending=False)
)

df_pass.write.format("delta").mode("overwrite").saveAsTable("pass_rates_by_program")
df_pass.show(truncate=False)
print("✓ pass_rates_by_program saved")

In [ ]:
# ── Table 3: score_distributions (percentiles for violin/box plots in PBI) ───
pcts = [0.10, 0.25, 0.50, 0.75, 0.90]
pct_labels = ["p10", "p25", "p50", "p75", "p90"]

rows = []
for dim in DIMENSIONS:
    for prefix in ["human", "model"]:
        col_name = f"{prefix}_{dim}"
        stats = df_eval.select(col_name).na.drop().approxQuantile(col_name, pcts, 0.01)
        row = {
            "dimension": dim,
            "scorer": prefix,
        }
        for label, val in zip(pct_labels, stats):
            row[label] = round(val, 3)
        rows.append(row)

dist_schema = StructType([
    StructField("dimension", StringType()),
    StructField("scorer", StringType()),
    *[StructField(p, DoubleType()) for p in pct_labels],
])
df_dist = spark.createDataFrame(rows, dist_schema)
df_dist.write.format("delta").mode("overwrite").saveAsTable("score_distributions")
df_dist.show(truncate=False)
print("✓ score_distributions saved")

In [ ]:
# ── Table 4: gate_confusion (TP/TN/FP/FN by program area) ───────────────────
df_conf = (
    df_eval
    .groupBy("program_area", "confusion_cell")
    .count()
    .withColumnRenamed("count", "n")
)

df_conf_pivot = (
    df_conf
    .groupBy("program_area")
    .pivot("confusion_cell", ["TP", "TN", "FP", "FN"])
    .sum("n")
    .na.fill(0)
    .withColumn(
        "accuracy",
        F.round((F.col("TP") + F.col("TN")) / (F.col("TP") + F.col("TN") + F.col("FP") + F.col("FN")), 3)
    )
    .withColumn(
        "false_positive_rate",
        F.when(
            (F.col("FP") + F.col("TN")) > 0,
            F.round(F.col("FP") / (F.col("FP") + F.col("TN")), 3)
        ).otherwise(F.lit(None))
    )
    .orderBy("false_positive_rate", ascending=False)
)

df_conf_pivot.write.format("delta").mode("overwrite").saveAsTable("gate_confusion")
df_conf_pivot.show(truncate=False)
print("✓ gate_confusion saved")

In [ ]:
# ── Table 5: calibration_summary (from calibration_report.json) ──────────────
try:
    cal_json = spark.read.text(CALIBRATION_PATH).collect()
    cal_text = "\n".join(r.value for r in cal_json)
    cal = json.loads(cal_text)

    cal_rows = []
    for dim, stats in cal.get("dimension_calibration", {}).items():
        cal_rows.append({
            "dimension": dim,
            "n": int(stats.get("n", 0)),
            "mean_delta": float(stats.get("mean_delta", 0.0)),
            "mae": float(stats.get("mae", 0.0)),
            "std_delta": float(stats.get("std_delta", 0.0)),
            "bias_direction": stats.get("bias_direction", ""),
            "human_mean": float(stats.get("human_mean") or 0.0),
            "model_mean": float(stats.get("model_mean") or 0.0),
        })

    cal_schema = StructType([
        StructField("dimension", StringType()),
        StructField("n", IntegerType()),
        StructField("mean_delta", DoubleType()),
        StructField("mae", DoubleType()),
        StructField("std_delta", DoubleType()),
        StructField("bias_direction", StringType()),
        StructField("human_mean", DoubleType()),
        StructField("model_mean", DoubleType()),
    ])
    df_cal = spark.createDataFrame(cal_rows, cal_schema)
    df_cal.write.format("delta").mode("overwrite").saveAsTable("calibration_summary")
    df_cal.show(truncate=False)
    print("✓ calibration_summary saved")
except Exception as e:
    print(f"⚠ calibration_report.json not found: {e}")
    print("  Run eval_pairs.py first to generate calibration_report.json")

In [ ]:
# ── Table 6: eval_timeline (scores over time — trend analysis in Power BI) ───
df_timeline = (
    df_eval
    .withColumn("annotation_month", F.date_trunc("month", F.col("annotated_at")))
    .groupBy("annotation_month")
    .agg(
        F.count("*").alias("total"),
        F.round(F.avg("human_groundedness"), 3).alias("avg_groundedness"),
        F.round(F.avg("human_completeness"), 3).alias("avg_completeness"),
        F.round(F.avg("human_compliance"), 3).alias("avg_compliance"),
        F.round(F.avg("human_parameter_accuracy"), 3).alias("avg_parameter_accuracy"),
        F.round(F.avg("human_coherence"), 3).alias("avg_coherence"),
        F.sum(F.when(F.col("human_gate") == "PASS", 1).otherwise(0)).alias("pass_count"),
    )
    .withColumn("pass_rate", F.round(F.col("pass_count") / F.col("total"), 3))
    .orderBy("annotation_month")
)

df_timeline.write.format("delta").mode("overwrite").saveAsTable("eval_timeline")
df_timeline.show(truncate=False)
print("✓ eval_timeline saved")

## Power BI Integration

All Delta tables above are auto-discoverable by Power BI via the Fabric Lakehouse semantic model.

### Suggested Power BI visuals

| Page | Visual | Table |
|---|---|---|
| Overview | Pass rate card + bar chart by program | `pass_rates_by_program` |
| Calibration | Scatter: human vs model score per dimension | `eval_results` |
| Gate Health | Confusion matrix heatmap by program area | `gate_confusion` |
| Score Drift | Box-whisker: p10/p25/p50/p75/p90 | `score_distributions` |
| Trend | Line chart pass_rate over time | `eval_timeline` |
| Bias Report | Bar: mean_delta + MAE per dimension | `calibration_summary` |

### Refresh schedule
Run this notebook after each annotation session or after `azd deploy` to keep analytics current.
Set a Fabric Data Pipeline to trigger it weekly or on `eval_pairs.jsonl` file change in OneLake.